In [1]:
import json
with open('company_tickers.json', 'r') as f:
    data = json.load(f)
cik = '320193'
for item in data.values():
    if str(item['cik_str']) == cik:
        print(f"CIK {cik}: {item['title']} (Ticker: {item['ticker']})")

CIK 320193: Apple Inc. (Ticker: AAPL)


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('sp500_final_common_cleaned.csv', dtype={'FIGI': str})
df_matching = pd.read_csv('matching_with_sector.csv', dtype={'FIGI': str})
cusip_to_ticker = dict(zip(df_matching['CUSIP'], df_matching['ticker']))
ticker_to_sector = dict(zip(df_matching['ticker'], df_matching['sector']))
sectors = sorted(set(df_matching['sector'].unique()) - {'Unknown'})
# Function to parse source_file to datetime for sorting
def parse_source_file(source_file):
    # Extract year and quarter (e.g., '2013q2_form13f.csv' -> '2013Q2')
    year = int(source_file[:4])
    quarter = source_file[4:6].lower()
    quarter_map = {'q1': '03-31', 'q2': '06-30', 'q3': '09-30', 'q4': '12-31'}
    date_str = f"{year}-{quarter_map[quarter]}"
    return pd.to_datetime(date_str)
# Sort by CIK and source_file (parsed as datetime)
df['source_file_dt'] = df['source_file'].apply(parse_source_file)
df = df.sort_values(['CIK', 'source_file_dt'])

cik_features = []
for cik, cik_group in df.groupby('CIK'):
    source_files = sorted(cik_group['source_file'].unique(), key=lambda x: parse_source_file(x))
    for source_file, group in cik_group.groupby('source_file'):
        features = {}
        features['CIK'] = cik
        features['source_file'] = source_file
        features['num_stocks'] = len(group['CUSIP'].unique())
        features['AUM'] = group['VALUE'].sum()
        # Compute HHI
        if features['AUM'] > 0:
            weights = group['VALUE'] / features['AUM']
            hhi = (weights ** 2).sum()
        else:
            hhi = 0
        features['HHI'] = hhi
        group['sector'] = group['CUSIP'].map(cusip_to_ticker).map(ticker_to_sector).fillna('Unknown')
        sector_values = group.groupby('sector')['VALUE'].sum()
        for sector in sectors:
            sector_value = sector_values.get(sector, 0)
            features[f'sector_{sector.replace(" ", "_")}'] = sector_value / features['AUM'] if features['AUM'] > 0 else 0
        # Compute turnover ratio
        current_idx = source_files.index(source_file) if source_file in source_files else -1
        if current_idx > 0:  # Previous period exists
            prev_source_file = source_files[current_idx - 1]
            prev_group = cik_group[cik_group['source_file'] == prev_source_file]
            prev_aum = prev_group['VALUE'].sum() * 1000
            # Merge current and previous holdings
            current_holdings = group[['CUSIP', 'VALUE']].set_index('CUSIP')['VALUE']
            prev_holdings = prev_group[['CUSIP', 'VALUE']].set_index('CUSIP')['VALUE']
            all_cusips = set(current_holdings.index).union(set(prev_holdings.index))
            value_diff = 0
            for cusip in all_cusips:
                curr_value = current_holdings.get(cusip, 0)
                prev_value = prev_holdings.get(cusip, 0)
                value_diff += abs(curr_value - prev_value)
            avg_aum = (features['AUM'] + prev_aum) / 2 if prev_aum > 0 else features['AUM']
            features['turnover_ratio'] = value_diff / avg_aum if avg_aum > 0 else 0
            # Compute Historical Similarity Score
            if features['AUM'] > 0 and prev_aum > 0:
                # Get weights for current and previous periods
                curr_weights = current_holdings.reindex(all_cusips, fill_value=0) / features['AUM']
                prev_weights = prev_holdings.reindex(all_cusips, fill_value=0) / prev_aum
                # Cosine similarity
                dot_product = np.dot(curr_weights, prev_weights)
                norm_curr = np.linalg.norm(curr_weights)
                norm_prev = np.linalg.norm(prev_weights)
                similarity = dot_product / (norm_curr * norm_prev) if norm_curr > 0 and norm_prev > 0 else 0
                features['historical_similarity'] = similarity
            else:
                features['historical_similarity'] = 0
        else:
            features['turnover_ratio'] = 0  # No previous period
            features['historical_similarity'] = 0  # No previous period
        if features['AUM'] > 0:
            # Sort by VALUE descending, take top min(10, num_stocks)
            top_holdings = group[['CUSIP', 'VALUE']].sort_values(by='VALUE', ascending=False)
            n = min(10, features['num_stocks'])
            top_values = top_holdings['VALUE'].head(n)
            weights = top_values / top_values.sum()  # Normalize to sum to 1
            entropy = -np.sum(weights * np.log(weights + 1e-10))  # Add small constant to avoid log(0)
            features['top_holdings_entropy'] = entropy
        else:
            features['top_holdings_entropy'] = 0
        cik_features.append(features)
df_cik = pd.DataFrame(cik_features)
df_cik = df_cik.sort_values(by=['source_file', 'CIK'])
df_cik.to_csv('cik_features.csv', index=False)
df_cik.head(50)

,CIK,source_file,num_stocks,AUM,HHI,sector_Basic_Materials,sector_Communication_Services,sector_Consumer_Cyclical,sector_Consumer_Defensive,sector_Energy,sector_Financial_Services,sector_Healthcare,sector_Industrials,sector_Real_Estate,sector_Technology,sector_Utilities,turnover_ratio,historical_similarity,top_holdings_entropy
487,1103245,2013q2_form13f.csv,26,62641,0.044044,0.032199,0.000000,0.078623,0.146917,0.125222,0.096662,0.140020,0.096933,0.000000,0.280200,0.003225,0.000000,0.000000,2.292117e+00
0,53417,2013q3_form13f.csv,171,42929700,0.020963,0.010855,0.049071,0.189407,0.086611,0.057092,0.150131,0.207721,0.087859,0.012199,0.125646,0.023408,0.000000,0.000000,2.276766e+00
45,105495,2013q3_form13f.csv,137,1834507,0.020204,0.023392,0.023291,0.113192,0.120699,0.097680,0.123648,0.207367,0.137677,0.000113,0.147073,0.005869,0.000000,0.000000,2.259431e+00
95,723188,2013q3_form13f.csv,141,150070,0.040630,0.016779,0.074912,0.018258,0.160272,0.102466,0.037423,0.138915,0.157233,0.000360,0.215466,0.077917,0.000000,0.000000,2.125169e+00
97,883965,2013q3_form13f.csv,19,769049,0.095214,0.093030,0.137111,0.000000,0.073934,0.000000,0.310907,0.014960,0.105982,0.000000,0.264076,0.000000,0.000000,0.000000,2.206125e+00
144,903949,2013q3_form13f.csv,89,1381136,0.036618,0.005161,0.000077,0.218692,0.043866,0.051650,0.156996,0.214250,0.169560,0.052825,0.086910,0.000013,0.000000,0.000000,2.290437e+00
191,1009076,2013q3_form13f.csv,245,2363310,0.019781,0.008844,0.059707,0.054929,0.174437,0.165543,0.094766,0.151257,0.138518,0.003771,0.126978,0.021250,0.000000,0.000000,2.243860e+00
237,1010873,2013q3_form13f.csv,94,237464,0.022574,0.026669,0.040086,0.114392,0.083878,0.065324,0.164699,0.151800,0.115239,0.004510,0.215734,0.017670,0.000000,0.000000,2.277931e+00
284,1020585,2013q3_form13f.csv,190,157400,0.019006,0.031105,0.033755,0.046569,0.100616,0.118990,0.137211,0.207313,0.164504,0.019568,0.119485,0.020883,0.000000,0.000000,2.207206e+00
331,1054233,2013q3_form13f.csv,85,238818,0.029980,0.000000,0.031995,0.084085,0.252845,0.070522,0.084370,0.100675,0.166399,0.016762,0.124379,0.067968,0.000000,0.000000,2.273602e+00


In [4]:
import pandas as pd

df = pd.read_csv('cik_features.csv')
df.info()
print(df.isna().sum())
print(df.duplicated(subset=['CIK', 'source_file']).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2062 entries, 0 to 2061
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CIK                            2062 non-null   int64  
 1   source_file                    2062 non-null   object 
 2   num_stocks                     2062 non-null   int64  
 3   AUM                            2062 non-null   int64  
 4   HHI                            2062 non-null   float64
 5   sector_Basic_Materials         2062 non-null   float64
 6   sector_Communication_Services  2062 non-null   float64
 7   sector_Consumer_Cyclical       2062 non-null   float64
 8   sector_Consumer_Defensive      2062 non-null   float64
 9   sector_Energy                  2062 non-null   float64
 10  sector_Financial_Services      2062 non-null   float64
 11  sector_Healthcare              2062 non-null   float64
 12  sector_Industrials             2062 non-null   f

In [ ]:
import pandas as pd
from tgb.nodeproppred.dataset import NodePropPredDataset

dataset = NodePropPredDataset(name='tgbn-13f', root='datasets')
df = pd.read_csv('cik_features.csv')
dict_cik = dict(zip(df['source_file'].unique(), dataset.label_ts))
df['source_file'] = df['source_file'].map(dict_cik)
df.to_csv('cik_features.csv', index=False)
df.head(50)

raw file found, skipping download
Dataset directory is  c:\Users\emadi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tgb/datasets\tgbn_13f
loading processed file


,CIK,source_file,num_stocks,AUM,HHI,sector_Basic_Materials,sector_Communication_Services,sector_Consumer_Cyclical,sector_Consumer_Defensive,sector_Energy,sector_Financial_Services,sector_Healthcare,sector_Industrials,sector_Real_Estate,sector_Technology,sector_Utilities,turnover_ratio,historical_similarity,top_holdings_entropy
0,1103245,20132,26,62641,0.044044,0.032199,0.000000,0.078623,0.146917,0.125222,0.096662,0.140020,0.096933,0.000000,0.280200,0.003225,0.000000,0.000000,2.292117e+00
1,53417,20133,171,42929700,0.020963,0.010855,0.049071,0.189407,0.086611,0.057092,0.150131,0.207721,0.087859,0.012199,0.125646,0.023408,0.000000,0.000000,2.276766e+00
2,105495,20133,137,1834507,0.020204,0.023392,0.023291,0.113192,0.120699,0.097680,0.123648,0.207367,0.137677,0.000113,0.147073,0.005869,0.000000,0.000000,2.259431e+00
3,723188,20133,141,150070,0.040630,0.016779,0.074912,0.018258,0.160272,0.102466,0.037423,0.138915,0.157233,0.000360,0.215466,0.077917,0.000000,0.000000,2.125169e+00
4,883965,20133,19,769049,0.095214,0.093030,0.137111,0.000000,0.073934,0.000000,0.310907,0.014960,0.105982,0.000000,0.264076,0.000000,0.000000,0.000000,2.206125e+00
5,903949,20133,89,1381136,0.036618,0.005161,0.000077,0.218692,0.043866,0.051650,0.156996,0.214250,0.169560,0.052825,0.086910,0.000013,0.000000,0.000000,2.290437e+00
6,1009076,20133,245,2363310,0.019781,0.008844,0.059707,0.054929,0.174437,0.165543,0.094766,0.151257,0.138518,0.003771,0.126978,0.021250,0.000000,0.000000,2.243860e+00
7,1010873,20133,94,237464,0.022574,0.026669,0.040086,0.114392,0.083878,0.065324,0.164699,0.151800,0.115239,0.004510,0.215734,0.017670,0.000000,0.000000,2.277931e+00
8,1020585,20133,190,157400,0.019006,0.031105,0.033755,0.046569,0.100616,0.118990,0.137211,0.207313,0.164504,0.019568,0.119485,0.020883,0.000000,0.000000,2.207206e+00
9,1054233,20133,85,238818,0.029980,0.000000,0.031995,0.084085,0.252845,0.070522,0.084370,0.100675,0.166399,0.016762,0.124379,0.067968,0.000000,0.000000,2.273602e+00
